# Categorical Encoding Analysis

This notebook analyzes the categorical features in the IEEE-CIS Fraud Detection dataset.

The goal is to:
- understand the categorical columns in the transaction and identity datasets,
- check their missing values and number of unique categories,
- choose an appropriate encoding method for each feature or feature group.

The final encoding strategy will be used to build a reusable preprocessing module for the team's machine learning models.

In [5]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Set the dataset folder path.

dataset_path = "/content/drive/MyDrive/ieee-fraud-detection (1)"

os.listdir(dataset_path)

['sample_submission.csv',
 'test_identity.csv',
 'test_transaction.csv',
 'train_identity.csv',
 'train_transaction.csv']

In [9]:
# Load the identity training dataset.

identity_path = f"{dataset_path}/train_identity.csv"

identity_df = pd.read_csv(identity_path)

identity_df.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [10]:
# List the categorical features in the identity dataset.

identity_cat_cols = (
    [f"id_{i:02d}" for i in range(12, 39)]
    + ["DeviceType", "DeviceInfo"]
)

identity_cat_cols

['id_12',
 'id_13',
 'id_14',
 'id_15',
 'id_16',
 'id_17',
 'id_18',
 'id_19',
 'id_20',
 'id_21',
 'id_22',
 'id_23',
 'id_24',
 'id_25',
 'id_26',
 'id_27',
 'id_28',
 'id_29',
 'id_30',
 'id_31',
 'id_32',
 'id_33',
 'id_34',
 'id_35',
 'id_36',
 'id_37',
 'id_38',
 'DeviceType',
 'DeviceInfo']

In [11]:
# Check unique values and missing data for each categorical feature.

identity_summary = pd.DataFrame({
    "Feature": identity_cat_cols,
    "Unique Values": [
        identity_df[col].nunique(dropna=True)
        for col in identity_cat_cols
    ],
    "Missing Values": [
        identity_df[col].isna().sum()
        for col in identity_cat_cols
    ],
    "Missing %": [
        round(identity_df[col].isna().mean() * 100, 2)
        for col in identity_cat_cols
    ]
})

identity_summary

,Feature,Unique Values,Missing Values,Missing %
0,id_12,2,0,0.00
1,id_13,54,16913,11.73
2,id_14,25,64189,44.50
3,id_15,3,3248,2.25
4,id_16,2,14893,10.33
5,id_17,104,4864,3.37
6,id_18,18,99120,68.72
7,id_19,522,4915,3.41
8,id_20,394,4972,3.45
9,id_21,490,139074,96.42


In [12]:
# Show the values and counts for low-cardinality features.

low_cardinality_cols = [
    col for col in identity_cat_cols
    if identity_df[col].nunique(dropna=True) <= 10
]

for col in low_cardinality_cols:
    print(f"\n{col}")
    print(identity_df[col].value_counts(dropna=False))


id_12
id_12
NotFound    123025
Found        21208
Name: count, dtype: int64

id_15
id_15
Found      67728
New        61612
Unknown    11645
NaN         3248
Name: count, dtype: int64

id_16
id_16
Found       66324
NotFound    63016
NaN         14893
Name: count, dtype: int64

id_23
id_23
NaN                     139064
IP_PROXY:TRANSPARENT      3489
IP_PROXY:ANONYMOUS        1071
IP_PROXY:HIDDEN            609
Name: count, dtype: int64

id_27
id_27
NaN         139064
Found         5155
NotFound        14
Name: count, dtype: int64

id_28
id_28
Found    76232
New      64746
NaN       3255
Name: count, dtype: int64

id_29
id_29
Found       74926
NotFound    66052
NaN          3255
Name: count, dtype: int64

id_32
id_32
NaN     66647
24.0    53071
32.0    24428
16.0       81
0.0         6
Name: count, dtype: int64

id_34
id_34
NaN                66428
match_status:2     60011
match_status:1     17376
match_status:0       415
match_status:-1        3
Name: count, dtype: int64

id_35
id_35
T

In [13]:
# Show the most common values for medium-cardinality features.

medium_cardinality_cols = [
    col for col in identity_cat_cols
    if 10 < identity_df[col].nunique(dropna=True) <= 100
]

for col in medium_cardinality_cols:
    print(f"\n{col}")
    print(identity_df[col].value_counts(dropna=False).head(15))


id_13
id_13
52.0    58099
49.0    26365
NaN     16913
64.0    14429
33.0    10048
27.0     3666
20.0     2878
14.0     2499
63.0     1468
19.0     1147
25.0     1066
43.0      842
62.0      813
18.0      688
41.0      654
Name: count, dtype: int64

id_14
id_14
 NaN      64189
-300.0    44121
-360.0    16661
-480.0    12891
-420.0     4542
-600.0      498
 60.0       369
 0.0        192
-240.0      159
-180.0      126
-540.0      111
 480.0       80
 540.0       64
 600.0       62
 120.0       41
Name: count, dtype: int64

id_18
id_18
NaN     99120
15.0    25489
13.0    13439
12.0     4656
18.0      650
20.0      339
17.0      233
26.0       89
21.0       78
24.0       52
11.0       36
27.0       32
29.0        9
23.0        4
14.0        3
Name: count, dtype: int64

id_22
id_22
NaN     139064
14.0      4736
41.0       321
33.0        38
21.0         7
17.0         7
39.0         6
31.0         5
12.0         5
35.0         5
22.0         5
36.0         5
26.0         4
20.0         4


In [14]:
# Show the most common values for high-cardinality features.

high_cardinality_cols = [
    col for col in identity_cat_cols
    if identity_df[col].nunique(dropna=True) > 100
]

for col in high_cardinality_cols:
    print(f"\n{col}")
    print(identity_df[col].value_counts(dropna=False).head(15))


id_17
id_17
166.0    78631
225.0    56968
NaN       4864
102.0      689
159.0      352
100.0      336
121.0      279
148.0      229
150.0      126
191.0      123
142.0      122
192.0      108
144.0       93
149.0       84
218.0       77
Name: count, dtype: int64

id_19
id_19
266.0    19849
410.0    11318
427.0     8808
529.0     8122
312.0     6227
100.0     5262
542.0     5116
NaN       4915
215.0     4728
153.0     4384
417.0     4085
352.0     3916
176.0     3845
290.0     3618
193.0     3094
Name: count, dtype: int64

id_20
id_20
507.0    22311
222.0    11065
325.0     8133
533.0     6611
214.0     5664
549.0     5643
600.0     5563
NaN       4972
563.0     4711
333.0     3691
595.0     3478
161.0     3174
500.0     2804
401.0     2576
489.0     2550
Name: count, dtype: int64

id_21
id_21
NaN      139074
252.0      2542
228.0       239
255.0       109
596.0       103
576.0       101
849.0        88
277.0        86
755.0        65
848.0        58
668.0        53
770.0        49
249

## Identity Encoding Strategy

Based on the cardinality, missingness, and value distributions, the identity categorical features will not all use the same encoding method.

### Low-cardinality features
These features have only a small number of categories:

`id_12`, `id_15`, `id_16`, `id_23`, `id_27`, `id_28`, `id_29`,
`id_32`, `id_34`, `id_35`, `id_36`, `id_37`, `id_38`, `DeviceType`

**Plan:** Use one-hot encoding because these features have only a few possible values.

### Moderate-cardinality features
These features have more categories, but not enough to require heavy compression:

`id_13`, `id_14`, `id_18`, `id_22`, `id_24`

**Plan:** Use one-hot encoding because the number of categories remains manageable.

### Higher-cardinality coded features
These features contain many different category codes:

`id_17`, `id_19`, `id_20`, `id_21`, `id_25`, `id_26`

**Plan:** Use frequency encoding for the larger coded features. For features such as `id_17`, where only a few values dominate, rare values may first be grouped together.

### Structured text features
Some categorical features contain information that can be simplified before encoding:

- `id_30`: group operating-system versions into broader families such as Windows, iOS, MacOS, and Android.
- `id_31`: group browser versions into browser families such as Chrome, Safari, Edge, and Firefox.
- `id_33`: treat screen resolution as a high-cardinality categorical feature and use frequency encoding.
- `DeviceInfo`: values appearing at least 50 times will remain as separate categories, while less frequent values will be grouped into `OTHER` before one-hot encoding.

### Missing values
Missing categorical values will be kept and represented as a separate `MISSING` category instead of dropping those rows.

Some identity features are more than 90% missing. Whether those features should remain in the final model will be decided later during feature selection.

### Group Operating System Versions

`id_30` contains many operating system versions. These will be grouped into broader operating system families so that similar versions are treated as the same category.

In [15]:
# Group operating system versions into broader families.

def group_os(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).lower()

    if "windows" in value:
        return "Windows"
    elif "ios" in value:
        return "iOS"
    elif "mac" in value:
        return "MacOS"
    elif "android" in value:
        return "Android"
    elif "linux" in value:
        return "Linux"
    else:
        return "Other"

identity_df["id_30_grouped"] = identity_df["id_30"].apply(group_os)

identity_df["id_30_grouped"].value_counts()

,count
id_30_grouped,
MISSING,66668
Windows,36739
iOS,19782
MacOS,13580
Android,6303
Linux,1136
Other,25


### Group Browser Versions

`id_31` contains different browser names and versions. Similar versions will be grouped into broader browser families.

In [58]:
# Group browser versions into broader families.

def group_browser(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).lower()

    if "samsung" in value:
        return "Samsung"
    elif "chrome" in value and "chromium" not in value:
        return "Chrome"
    elif "chromium" in value:
        return "Chromium"
    elif "safari" in value:
        return "Safari"
    elif "firefox" in value:
        return "Firefox"
    elif "edge" in value:
        return "Edge"
    elif value == "ie" or value.startswith("ie ") or "internet explorer" in value:
        return "Internet Explorer"
    elif "android browser" in value or "generic/android" in value or value == "android":
        return "Android Browser"
    elif "google search application" in value or value == "google":
        return "Google Search"
    elif "silk" in value:
        return "Silk"
    elif "opera" in value:
        return "Opera"
    elif "android webview" in value:
        return "Android WebView"
    else:
        return "Other"

identity_df["id_31_grouped"] = identity_df["id_31"].apply(group_browser)

In [59]:
identity_df["id_31_grouped"].value_counts()

,count
id_31_grouped,
Chrome,76059
Safari,37281
Internet Explorer,9733
Firefox,7017
Edge,6401
MISSING,3951
Samsung,2247
Opera,449
Other,405


In [22]:
# Count every unique DeviceInfo value.

device_counts = (
    identity_df["DeviceInfo"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("DeviceInfo")
    .reset_index(name="Count")
)

device_counts

,DeviceInfo,Count
0,Windows,47722
1,MISSING,25567
2,iOS Device,19782
3,MacOS,12573
4,Trident/7.0,7440
...,...,...
1782,5044A,1
1783,LG-D373,1
1784,Bolt,1
1785,LG-K350,1


In [23]:
# Compare possible cutoffs for grouping rare DeviceInfo values.

device_counts = identity_df["DeviceInfo"].value_counts()

cutoffs = [5, 10, 20, 50, 100, 200]

cutoff_results = []

for cutoff in cutoffs:
    common = device_counts[device_counts >= cutoff]

    cutoff_results.append({
        "Minimum Count": cutoff,
        "Categories Kept": len(common),
        "Categories Grouped as OTHER": len(device_counts) - len(common),
        "Rows Kept in Original Category": common.sum(),
        "Rows Grouped as OTHER": device_counts[device_counts < cutoff].sum()
    })

pd.DataFrame(cutoff_results)

,Minimum Count,Categories Kept,Categories Grouped as OTHER,Rows Kept in Original Category,Rows Grouped as OTHER
0,5,887,899,116928,1738
1,10,513,1273,114484,4182
2,20,311,1475,111712,6954
3,50,147,1639,106782,11884
4,100,63,1723,100970,17696
5,200,23,1763,95632,23034


### DeviceInfo Rare Category Grouping

`DeviceInfo` contains 1,786 unique non-missing values, with many device names appearing only a few times.

A minimum count of 50 was selected. Device values that appear at least 50 times will remain as separate categories, while values appearing fewer than 50 times will be grouped into `OTHER`.

This reduces the number of categories while still keeping about 90% of non-missing rows in their original device category. Missing values will remain as `MISSING`.

In [24]:
# Keep common DeviceInfo values and group rare values into OTHER.

device_counts = identity_df["DeviceInfo"].value_counts()

common_devices = device_counts[
    device_counts >= 50
].index

def group_device_info(value):
    if pd.isna(value):
        return "MISSING"

    if value in common_devices:
        return value

    return "OTHER"

identity_df["DeviceInfo_grouped"] = (
    identity_df["DeviceInfo"].apply(group_device_info)
)

identity_df["DeviceInfo_grouped"].value_counts()

,count
DeviceInfo_grouped,
Windows,47722
MISSING,25567
iOS Device,19782
MacOS,12573
OTHER,11884
...,...
XT1032 Build/LPBS23.13-56-2,50
LG-H650 Build/MRA58K,50
LG-X220 Build/LMY47I,50


### Frequency Encoding High-Cardinality Features

Some identity features contain hundreds of category codes. Instead of creating hundreds of one-hot columns, these features will be replaced with how frequently each value appears.

Missing values will be treated as their own category.

In [25]:
# Frequency encode the high-cardinality coded features.

frequency_cols = [
    "id_19",
    "id_20",
    "id_21",
    "id_25",
    "id_26",
    "id_33"
]

for col in frequency_cols:
    values = identity_df[col].fillna("MISSING")
    frequencies = values.value_counts(normalize=True)

    identity_df[f"{col}_freq"] = values.map(frequencies)

identity_df[
    frequency_cols +
    [f"{col}_freq" for col in frequency_cols]
].head()

,id_19,id_20,id_21,id_25,id_26,id_33,id_19_freq,id_20_freq,id_21_freq,id_25_freq,id_26_freq,id_33_freq
0,542.0,144.0,NaN,NaN,NaN,2220x1080,0.035470,0.004680,0.964231,0.964419,0.964204,0.003772
1,621.0,500.0,NaN,NaN,NaN,1334x750,0.014927,0.019441,0.964231,0.964419,0.964204,0.044699
2,410.0,142.0,NaN,NaN,NaN,NaN,0.078470,0.001248,0.964231,0.964419,0.964204,0.491871
3,176.0,507.0,NaN,NaN,NaN,NaN,0.026658,0.154687,0.964231,0.964419,0.964204,0.491871
4,529.0,575.0,NaN,NaN,NaN,1280x800,0.056312,0.001511,0.964231,0.964419,0.964204,0.014900


In [26]:
# Show all id_17 values and how often each one appears.

id17_counts = (
    identity_df["id_17"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("id_17")
    .reset_index(name="Count")
)

id17_counts

,id_17,Count
0,166.0,78631
1,225.0,56968
2,MISSING,4864
3,102.0,689
4,159.0,352
...,...,...
100,160.0,1
101,139.0,1
102,211.0,1
103,188.0,1


In [27]:
# Compare possible cutoffs for grouping rare id_17 values.

id17_counts = identity_df["id_17"].value_counts()

cutoffs = [5, 10, 20, 50, 100]

cutoff_results = []

for cutoff in cutoffs:
    common = id17_counts[id17_counts >= cutoff]

    cutoff_results.append({
        "Minimum Count": cutoff,
        "Categories Kept": len(common),
        "Categories Grouped as OTHER": len(id17_counts) - len(common),
        "Rows Kept in Original Category": common.sum(),
        "Rows Grouped as OTHER": id17_counts[id17_counts < cutoff].sum()
    })

pd.DataFrame(cutoff_results)

,Minimum Count,Categories Kept,Categories Grouped as OTHER,Rows Kept in Original Category,Rows Grouped as OTHER
0,5,75,29,139315,54
1,10,51,53,139154,215
2,20,33,71,138911,458
3,50,17,87,138404,965
4,100,11,93,137963,1406


### Group Rare `id_17` Values

`id_17` has many categories, but most rows are concentrated in a small number of values.

Values appearing at least 50 times will remain as separate categories. Values appearing fewer than 50 times will be grouped into `OTHER`, while missing values will remain as `MISSING`.

In [28]:
# Group rare id_17 values into OTHER.

id17_counts = identity_df["id_17"].value_counts()

common_id17 = id17_counts[
    id17_counts >= 50
].index

def group_id17(value):
    if pd.isna(value):
        return "MISSING"

    if value in common_id17:
        return str(value)

    return "OTHER"

identity_df["id_17_grouped"] = (
    identity_df["id_17"].apply(group_id17)
)

identity_df["id_17_grouped"].value_counts()

,count
id_17_grouped,
166.0,78631
225.0,56968
MISSING,4864
OTHER,965
102.0,689
159.0,352
100.0,336
121.0,279
148.0,229


### One-Hot Encode Low and Moderate Cardinality Features

The remaining categorical features have a manageable number of categories. These features will use one-hot encoding.

For grouped features such as operating system, browser, and `id_17`, the cleaned versions will be used instead of the original columns. Missing values will be kept as their own category.

In [29]:
# List the identity features that will use one-hot encoding.

one_hot_cols = [
    "id_12",
    "id_13",
    "id_14",
    "id_15",
    "id_16",
    "id_18",
    "id_22",
    "id_23",
    "id_24",
    "id_27",
    "id_28",
    "id_29",
    "id_32",
    "id_34",
    "id_35",
    "id_36",
    "id_37",
    "id_38",
    "DeviceType",
    "id_17_grouped",
    "id_30_grouped",
    "id_31_grouped",
    "DeviceInfo_grouped"
]

In [30]:
# Replace missing values with an explicit MISSING category.

identity_one_hot = identity_df[one_hot_cols].copy()

for col in identity_one_hot.columns:
    identity_one_hot[col] = (
        identity_one_hot[col]
        .astype("object")
        .where(identity_one_hot[col].notna(), "MISSING")
        .astype(str)
    )

In [31]:
# One-hot encode the selected identity features.

identity_one_hot_encoded = pd.get_dummies(
    identity_one_hot,
    columns=one_hot_cols,
    dtype="int8"
)

identity_one_hot_encoded.shape

(144233, 375)

In [32]:
# Preview the one-hot encoded identity features.

identity_one_hot_encoded.head()

,id_12_Found,id_12_NotFound,id_13_10.0,id_13_11.0,id_13_12.0,id_13_13.0,id_13_14.0,id_13_15.0,id_13_17.0,id_13_18.0,...,DeviceInfo_grouped_hi6210sft Build/MRA58K,DeviceInfo_grouped_iOS Device,DeviceInfo_grouped_rv:11.0,DeviceInfo_grouped_rv:48.0,DeviceInfo_grouped_rv:52.0,DeviceInfo_grouped_rv:56.0,DeviceInfo_grouped_rv:57.0,DeviceInfo_grouped_rv:58.0,DeviceInfo_grouped_rv:59.0,DeviceInfo_grouped_rv:60.0
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Final Encoded Identity Features

The one-hot and frequency-encoded features are combined into one dataset. `TransactionID` is kept so the encoded features can later be matched with the transaction data.

In [33]:
# Combine all encoded identity features.

frequency_encoded_cols = [
    "id_19_freq",
    "id_20_freq",
    "id_21_freq",
    "id_25_freq",
    "id_26_freq",
    "id_33_freq"
]

identity_encoded = pd.concat(
    [
        identity_df[["TransactionID"]].reset_index(drop=True),
        identity_one_hot_encoded.reset_index(drop=True),
        identity_df[frequency_encoded_cols].reset_index(drop=True)
    ],
    axis=1
)

identity_encoded.shape

(144233, 382)

In [34]:
# Preview the final encoded identity features.

identity_encoded.head()

,TransactionID,id_12_Found,id_12_NotFound,id_13_10.0,id_13_11.0,id_13_12.0,id_13_13.0,id_13_14.0,id_13_15.0,id_13_17.0,...,DeviceInfo_grouped_rv:57.0,DeviceInfo_grouped_rv:58.0,DeviceInfo_grouped_rv:59.0,DeviceInfo_grouped_rv:60.0,id_19_freq,id_20_freq,id_21_freq,id_25_freq,id_26_freq,id_33_freq
0,2987004,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0.035470,0.004680,0.964231,0.964419,0.964204,0.003772
1,2987008,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0.014927,0.019441,0.964231,0.964419,0.964204,0.044699
2,2987010,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0.078470,0.001248,0.964231,0.964419,0.964204,0.491871
3,2987011,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0.026658,0.154687,0.964231,0.964419,0.964204,0.491871
4,2987016,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0.056312,0.001511,0.964231,0.964419,0.964204,0.014900


## Identity Encoding Summary

The categorical identity features were encoded using different methods based on their structure and cardinality.

- Low- and moderate-cardinality features were one-hot encoded.
- Rare values in `id_17` and `DeviceInfo` were grouped into `OTHER`.
- `id_30` was grouped into operating system families.
- `id_31` was grouped into browser families.
- High-cardinality coded features (`id_19`, `id_20`, `id_21`, `id_25`, `id_26`, `id_33`) were frequency encoded.
- Missing categorical values were kept as `MISSING`.
- `TransactionID` was preserved so the encoded identity features can later be matched back to the transaction data.

The final encoded identity dataset contains 144,233 rows and 382 columns.

## Transaction Categorical Feature Analysis

This section examines the categorical features in the transaction dataset. The goal is to check missing values, number of unique categories, and value distributions before choosing an encoding strategy.

In [37]:
# Load the categorical features from the transaction dataset.

transaction_cat_cols = [
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9"
]

transaction_path = f"{dataset_path}/train_transaction.csv"

transaction_df = pd.read_csv(
    transaction_path,
    usecols=transaction_cat_cols
)

transaction_df.head()

,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9
0,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,NaN,NaN,T,T,T,M2,F,T,NaN,NaN,NaN
1,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,gmail.com,NaN,NaN,NaN,NaN,M0,T,T,NaN,NaN,NaN
2,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,outlook.com,NaN,T,T,T,M0,F,F,F,F,F
3,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,yahoo.com,NaN,NaN,NaN,NaN,M0,T,F,NaN,NaN,NaN
4,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,gmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# Check unique values and missing data for each transaction categorical feature.

transaction_summary = pd.DataFrame({
    "Feature": transaction_cat_cols,
    "Unique Values": [
        transaction_df[col].nunique(dropna=True)
        for col in transaction_cat_cols
    ],
    "Missing Values": [
        transaction_df[col].isna().sum()
        for col in transaction_cat_cols
    ],
    "Missing %": [
        round(transaction_df[col].isna().mean() * 100, 2)
        for col in transaction_cat_cols
    ]
})

transaction_summary


,Feature,Unique Values,Missing Values,Missing %
0,ProductCD,5,0,0.00
1,card1,13553,0,0.00
2,card2,500,8933,1.51
3,card3,114,1565,0.27
4,card4,4,1577,0.27
5,card5,119,4259,0.72
6,card6,4,1571,0.27
7,addr1,332,65706,11.13
8,addr2,74,65706,11.13
9,P_emaildomain,59,94456,15.99


In [39]:
# Check the number of rows and categorical columns loaded.

transaction_df.shape

(590540, 20)

In [40]:
# Show values and counts for low-cardinality transaction features.

low_transaction_cols = [
    "ProductCD",
    "card4",
    "card6",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9"
]

for col in low_transaction_cols:
    print(f"\n{col}")
    print(transaction_df[col].value_counts(dropna=False))


ProductCD
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64

card6
card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64

M1
M1
T      319415
NaN    271100
F          25
Name: count, dtype: int64

M2
M2
T      285468
NaN    271100
F       33972
Name: count, dtype: int64

M3
M3
NaN    271100
T      251731
F       67709
Name: count, dtype: int64

M4
M4
NaN    281444
M0     196405
M2      59865
M1      52826
Name: count, dtype: int64

M5
M5
NaN    350482
F      132491
T      107567
Name: count, dtype: int64

M6
M6
F      227856
T      193324
NaN    169360
Name: count, dtype: int64

M7
M7
NaN    346265
F      211374
T       32901
Name: count, dtype: int64

M8
M8
NaN 

### Low-Cardinality Transaction Features

The low-cardinality transaction features contain only a few possible values, so they will use one-hot encoding.

- `ProductCD`, `card4`, and `M1`–`M9` will be one-hot encoded.
- Rare `card6` values will be grouped into `OTHER` before one-hot encoding.
- Missing values will be kept as a separate `MISSING` category.

In [41]:
# Group very rare card6 categories into OTHER.

def group_card6(value):
    if pd.isna(value):
        return "MISSING"

    if value in ["debit", "credit"]:
        return value

    return "OTHER"

transaction_df["card6_grouped"] = transaction_df["card6"].apply(group_card6)

transaction_df["card6_grouped"].value_counts()

,count
card6_grouped,
debit,439938
credit,148986
MISSING,1571
OTHER,45


In [42]:
# Show the most common values for medium-cardinality transaction features.

medium_transaction_cols = [
    "addr2",
    "P_emaildomain",
    "R_emaildomain"
]

for col in medium_transaction_cols:
    print(f"\n{col}")
    print(transaction_df[col].value_counts(dropna=False).head(30))


addr2
addr2
87.0     520481
NaN       65706
60.0       3084
96.0        638
32.0         91
65.0         82
16.0         55
31.0         47
19.0         33
26.0         25
27.0         20
69.0         17
59.0         17
34.0         16
43.0         12
102.0        11
29.0         11
98.0         11
57.0         10
68.0         10
78.0          8
10.0          8
13.0          7
71.0          7
17.0          7
54.0          6
72.0          6
88.0          5
52.0          5
21.0          5
Name: count, dtype: int64

P_emaildomain
P_emaildomain
gmail.com         228355
yahoo.com         100934
NaN                94456
hotmail.com        45250
anonymous.com      36998
aol.com            28289
comcast.net         7888
icloud.com          6267
outlook.com         5096
msn.com             4092
att.net             4033
live.com            3041
sbcglobal.net       2970
verizon.net         2705
ymail.com           2396
bellsouth.net       1909
yahoo.com.mx        1543
me.com              1522
co

### Medium-Cardinality Transaction Features

`addr2`, `P_emaildomain`, and `R_emaildomain` will use one-hot encoding.

Their number of categories is still manageable, and preserving the original category may retain useful information. Missing values will be represented as `MISSING`.

In [43]:
# Show the most common values for high-cardinality transaction features.

high_transaction_cols = [
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1"
]

for col in high_transaction_cols:
    print(f"\n{col}")
    print(transaction_df[col].value_counts(dropna=False).head(30))


card1
card1
7919     14932
9500     14162
15885    10361
17188    10344
15066     7945
12695     7091
12544     6773
6019      6771
2803      6141
7585      5334
10616     5172
12839     5129
3154      4614
2616      4410
18132     4209
9633      4158
15497     3977
16132     3929
2884      3873
16075     3748
11207     3693
10112     3560
7508      3490
10057     3166
12501     3152
7826      3006
16659     2988
12577     2891
5812      2818
7664      2792
Name: count, dtype: int64

card2
card2
321.0    48935
111.0    45191
555.0    41995
490.0    38145
583.0    21803
170.0    18214
194.0    16938
545.0    16355
360.0    15190
514.0    14541
174.0    11310
512.0    10126
NaN       8933
408.0     8012
361.0     7827
100.0     7570
225.0     7445
215.0     7281
399.0     7180
553.0     6495
481.0     6336
268.0     6239
567.0     6137
476.0     5822
375.0     5473
543.0     5451
327.0     5100
500.0     5045
298.0     4356
206.0     4208
Name: count, dtype: int64

card3
card3
150.0    

In [44]:
# Check how concentrated the high-cardinality features are.

for col in high_transaction_cols:
    counts = transaction_df[col].fillna("MISSING").value_counts()
    total = len(transaction_df)

    print(f"\n{col}")
    print("Unique categories:", len(counts))
    print("Top 10 coverage:", round(counts.head(10).sum() / total * 100, 2), "%")
    print("Top 50 coverage:", round(counts.head(50).sum() / total * 100, 2), "%")
    print("Top 100 coverage:", round(counts.head(100).sum() / total * 100, 2), "%")


card1
Unique categories: 13553
Top 10 coverage: 15.22 %
Top 50 coverage: 35.76 %
Top 100 coverage: 47.43 %

card2
Unique categories: 501
Top 10 coverage: 46.96 %
Top 50 coverage: 79.69 %
Top 100 coverage: 88.02 %

card3
Unique categories: 115
Top 10 coverage: 99.29 %
Top 50 coverage: 99.95 %
Top 100 coverage: 100.0 %

card5
Unique categories: 120
Top 10 coverage: 94.64 %
Top 50 coverage: 99.92 %
Top 100 coverage: 100.0 %

addr1
Unique categories: 333
Top 10 coverage: 58.1 %
Top 50 coverage: 98.69 %
Top 100 coverage: 99.9 %


In [45]:
# Compare rare-category cutoffs for selected high-cardinality features.

grouping_cols = ["card2", "card3", "card5", "addr1"]
cutoffs = [50, 100, 250, 500, 1000]

for col in grouping_cols:
    counts = transaction_df[col].value_counts()

    print(f"\n{col}")

    for cutoff in cutoffs:
        kept = (counts >= cutoff).sum()
        grouped = (counts < cutoff).sum()
        rows_kept = counts[counts >= cutoff].sum()
        rows_other = counts[counts < cutoff].sum()

        print(
            f"Cutoff {cutoff}: "
            f"{kept} kept, "
            f"{grouped} grouped, "
            f"{rows_kept} rows kept, "
            f"{rows_other} rows OTHER"
        )


card2
Cutoff 50: 488 kept, 12 grouped, 581156 rows kept, 451 rows OTHER
Cutoff 100: 363 kept, 137 grouped, 571299 rows kept, 10308 rows OTHER
Cutoff 250: 194 kept, 306 grouped, 544749 rows kept, 36858 rows OTHER
Cutoff 500: 107 kept, 393 grouped, 515148 rows kept, 66459 rows OTHER
Cutoff 1000: 68 kept, 432 grouped, 487307 rows kept, 94300 rows OTHER

card3
Cutoff 50: 33 kept, 81 grouped, 588137 rows kept, 838 rows OTHER
Cutoff 100: 23 kept, 91 grouped, 587451 rows kept, 1524 rows OTHER
Cutoff 250: 12 kept, 102 grouped, 585761 rows kept, 3214 rows OTHER
Cutoff 500: 8 kept, 106 grouped, 584319 rows kept, 4656 rows OTHER
Cutoff 1000: 5 kept, 109 grouped, 581708 rows kept, 7267 rows OTHER

card5
Cutoff 50: 44 kept, 75 grouped, 585587 rows kept, 694 rows OTHER
Cutoff 100: 37 kept, 82 grouped, 585045 rows kept, 1236 rows OTHER
Cutoff 250: 30 kept, 89 grouped, 583610 rows kept, 2671 rows OTHER
Cutoff 500: 22 kept, 97 grouped, 580831 rows kept, 5450 rows OTHER
Cutoff 1000: 15 kept, 104 groupe

### Rare Category Grouping

Rare categories are grouped into `OTHER` before one-hot encoding.

Different thresholds are used for each feature based on its distribution:

- `card2`: minimum count of 100
- `card3`: minimum count of 50
- `card5`: minimum count of 50
- `addr1`: minimum count of 50

Missing values are kept separately as `MISSING`.

In [46]:
# Group rare categories using the selected cutoffs.

grouping_cutoffs = {
    "card2": 100,
    "card3": 50,
    "card5": 50,
    "addr1": 50
}

for col, cutoff in grouping_cutoffs.items():
    counts = transaction_df[col].value_counts()
    common_values = counts[counts >= cutoff].index

    transaction_df[f"{col}_grouped"] = transaction_df[col].apply(
        lambda x: "MISSING"
        if pd.isna(x)
        else str(x)
        if x in common_values
        else "OTHER"
    )

In [47]:
# Check the grouped transaction features.

for col in grouping_cutoffs:
    grouped_col = f"{col}_grouped"

    print(f"\n{grouped_col}")
    print("Categories:", transaction_df[grouped_col].nunique())
    print(transaction_df[grouped_col].value_counts().head(10))


card2_grouped
Categories: 365
card2_grouped
321.0    48935
111.0    45191
555.0    41995
490.0    38145
583.0    21803
170.0    18214
194.0    16938
545.0    16355
360.0    15190
514.0    14541
Name: count, dtype: int64

card3_grouped
Categories: 35
card3_grouped
150.0      521287
185.0       56346
106.0        1571
MISSING      1565
146.0        1252
144.0        1252
117.0         962
143.0         899
OTHER         838
119.0         750
Name: count, dtype: int64

card5_grouped
Categories: 46
card5_grouped
226.0    296546
224.0     81513
166.0     57140
102.0     29105
117.0     25941
138.0     19737
195.0     16945
137.0     11720
126.0     10298
219.0      9924
Name: count, dtype: int64

addr1_grouped
Categories: 72
addr1_grouped
MISSING    65706
299.0      46335
325.0      42751
204.0      42020
264.0      39870
330.0      26287
315.0      23078
441.0      20827
272.0      20141
123.0      16105
Name: count, dtype: int64


### card1 Frequency Encoding

`card1` contains 13,553 unique category codes and has a broad distribution. One-hot encoding would create thousands of columns.

Instead, `card1` will use frequency encoding. In this exploratory analysis, each category is replaced by its frequency in the full training file. In the reusable encoder, the frequency mapping will be learned from the chronological training split only.

In [48]:
# Frequency encode card1.

card1_values = transaction_df["card1"].fillna("MISSING")
card1_frequencies = card1_values.value_counts(normalize=True)

transaction_df["card1_freq"] = card1_values.map(card1_frequencies)

transaction_df[["card1", "card1_freq"]].head(10)

,card1,card1_freq
0,13926,0.000073
1,2755,0.001157
2,4663,0.001876
3,18132,0.007127
4,4497,0.000030
5,5937,0.000012
6,12308,0.000351
7,12695,0.012008
8,2803,0.010399
9,17399,0.003244


### One-Hot Encode Transaction Features

The remaining categorical transaction features have a manageable number of categories after grouping. These features will now be one-hot encoded.

Grouped versions of `card2`, `card3`, `card5`, `card6`, and `addr1` are used instead of their original columns. Missing values are represented as `MISSING`.

In [49]:
# List the transaction features that will use one-hot encoding.

transaction_one_hot_cols = [
    "ProductCD",
    "card4",
    "card6_grouped",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9",
    "card2_grouped",
    "card3_grouped",
    "card5_grouped",
    "addr1_grouped"
]

In [50]:
# Replace missing values with the MISSING category.

transaction_one_hot = transaction_df[transaction_one_hot_cols].copy()

for col in transaction_one_hot.columns:
    transaction_one_hot[col] = (
        transaction_one_hot[col]
        .astype("object")
        .where(transaction_one_hot[col].notna(), "MISSING")
        .astype(str)
    )

In [51]:
# One-hot encode the selected transaction features.

transaction_one_hot_encoded = pd.get_dummies(
    transaction_one_hot,
    columns=transaction_one_hot_cols,
    dtype="int8"
)

transaction_one_hot_encoded.shape

(590540, 756)

In [52]:
# Load TransactionID so encoded features can be matched later.

transaction_ids = pd.read_csv(
    transaction_path,
    usecols=["TransactionID"]
)

transaction_ids.head()

,TransactionID
0,2987000
1,2987001
2,2987002
3,2987003
4,2987004


In [53]:
# Combine all encoded transaction categorical features.

transaction_encoded = pd.concat(
    [
        transaction_ids.reset_index(drop=True),
        transaction_one_hot_encoded.reset_index(drop=True),
        transaction_df[["card1_freq"]].reset_index(drop=True)
    ],
    axis=1
)

transaction_encoded.shape

(590540, 758)

In [54]:
# Preview the final encoded transaction features.

transaction_encoded.head()

,TransactionID,ProductCD_C,ProductCD_H,ProductCD_R,ProductCD_S,ProductCD_W,card4_MISSING,card4_american express,card4_discover,card4_mastercard,...,addr1_grouped_494.0,addr1_grouped_498.0,addr1_grouped_502.0,addr1_grouped_508.0,addr1_grouped_511.0,addr1_grouped_512.0,addr1_grouped_536.0,addr1_grouped_MISSING,addr1_grouped_OTHER,card1_freq
0,2987000,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0.000073
1,2987001,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0.001157
2,2987002,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.001876
3,2987003,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0.007127
4,2987004,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0.000030


## Transaction Encoding Summary

The categorical transaction features were encoded using different methods based on their cardinality and distributions.

- `card1` was frequency encoded because it contains 13,553 categories and has a large long tail.
- Rare values in `card2`, `card3`, `card5`, and `addr1` were grouped into `OTHER` before one-hot encoding.
- `card6` rare values were grouped into `OTHER`.
- Low- and moderate-cardinality features such as `ProductCD`, `card4`, `addr2`, the email domains, and `M1`–`M9` were one-hot encoded.
- Missing categorical values were represented as `MISSING`.
- `TransactionID` was preserved so the encoded features can later be joined with the remaining model features.

The final exploratory transaction categorical dataset contains 590,540 rows and 758 columns.

# Reusable Categorical Encoder

The exploratory analysis above was used to choose the encoding strategy. The final encoder learns category groupings, frequency mappings, and one-hot categories only from the training split.

The fitted mappings are then reused unchanged for validation and test data to prevent data leakage.

In [55]:
# Import tools for the reusable encoder.

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

In [56]:
# Store the final encoding strategy.

transaction_group_cutoffs = {
    "card2": 100,
    "card3": 50,
    "card5": 50,
    "addr1": 50
}

identity_group_cutoffs = {
    "id_17": 50,
    "DeviceInfo": 50
}

frequency_cols = [
    "card1",
    "id_19",
    "id_20",
    "id_21",
    "id_25",
    "id_26",
    "id_33"
]

In [57]:
# Store the features that will use one-hot encoding.

one_hot_cols = [
    "ProductCD",
    "card4",
    "card6_grouped",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9",

    "card2_grouped",
    "card3_grouped",
    "card5_grouped",
    "addr1_grouped",

    "id_12",
    "id_13",
    "id_14",
    "id_15",
    "id_16",
    "id_18",
    "id_22",
    "id_23",
    "id_24",
    "id_27",
    "id_28",
    "id_29",
    "id_32",
    "id_34",
    "id_35",
    "id_36",
    "id_37",
    "id_38",
    "DeviceType",

    "id_17_grouped",
    "id_30_grouped",
    "id_31_grouped",
    "DeviceInfo_grouped"
]

### CategoricalEncoder Class

The reusable encoder stores the selected encoding rules and learns category mappings only when `fit()` is called on the training split.

In [66]:
# Create the complete reusable categorical encoder.

class CategoricalEncoder:
    def __init__(self):
        # Encoding decisions chosen during exploratory analysis.
        self.transaction_group_cutoffs = transaction_group_cutoffs.copy()
        self.identity_group_cutoffs = identity_group_cutoffs.copy()
        self.frequency_cols = frequency_cols.copy()
        self.one_hot_cols = one_hot_cols.copy()

        # These are learned only from the training split.
        self.common_values_ = {}
        self.frequency_maps_ = {}
        self.one_hot_encoder_ = None

        self.is_fitted_ = False


    def _apply_fixed_groups(self, df):
        """Apply grouping rules that do not need to be learned."""
        data = df.copy()

        data["id_30_grouped"] = data["id_30"].apply(group_os)
        data["id_31_grouped"] = data["id_31"].apply(group_browser)
        data["card6_grouped"] = data["card6"].apply(group_card6)

        return data


    def fit(self, train_df):
        """Learn category mappings from the training split only."""
        data = self._apply_fixed_groups(train_df)

        # Combine all features that use frequency-based rare grouping.
        all_cutoffs = {
            **self.transaction_group_cutoffs,
            **self.identity_group_cutoffs
        }

        # Learn which categories are common enough to keep.
        for col, cutoff in all_cutoffs.items():
            counts = data[col].value_counts(dropna=True)

            self.common_values_[col] = set(
                counts[counts >= cutoff].index
            )

            data[f"{col}_grouped"] = data[col].apply(
                lambda x: (
                    "MISSING"
                    if pd.isna(x)
                    else str(x)
                    if x in self.common_values_[col]
                    else "OTHER"
                )
            )

        # Learn frequency mappings from training only.
        for col in self.frequency_cols:
            values = (
                data[col]
                .astype("object")
                .where(data[col].notna(), "MISSING")
                .astype(str)
            )

            self.frequency_maps_[col] = (
                values
                .value_counts(normalize=True)
                .to_dict()
            )

        # Prepare the features that will be one-hot encoded.
        one_hot_data = data[self.one_hot_cols].copy()

        for col in one_hot_data.columns:
            one_hot_data[col] = (
                one_hot_data[col]
                .astype("object")
                .where(one_hot_data[col].notna(), "MISSING")
                .astype(str)
            )

        # Learn one-hot categories from training only.
        self.one_hot_encoder_ = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )

        self.one_hot_encoder_.fit(one_hot_data)

        self.is_fitted_ = True

        return self


    def transform(self, df):
        """Apply learned training mappings to new data."""
        if not self.is_fitted_:
            raise ValueError(
                "Encoder must be fitted before transform()."
            )

        data = self._apply_fixed_groups(df)

        all_cutoffs = {
            **self.transaction_group_cutoffs,
            **self.identity_group_cutoffs
        }

        # Apply rare-category groups learned during fit().
        for col in all_cutoffs:
            data[f"{col}_grouped"] = data[col].apply(
                lambda x: (
                    "MISSING"
                    if pd.isna(x)
                    else str(x)
                    if x in self.common_values_[col]
                    else "OTHER"
                )
            )

        # Apply frequency mappings learned during fit().
        frequency_data = pd.DataFrame(index=data.index)

        for col in self.frequency_cols:
            values = (
                data[col]
                .astype("object")
                .where(data[col].notna(), "MISSING")
                .astype(str)
            )

            frequency_data[f"{col}_freq"] = (
                values
                .map(self.frequency_maps_[col])
                .fillna(0)
            )

        # Prepare one-hot features.
        one_hot_data = data[self.one_hot_cols].copy()

        for col in one_hot_data.columns:
            one_hot_data[col] = (
                one_hot_data[col]
                .astype("object")
                .where(one_hot_data[col].notna(), "MISSING")
                .astype(str)
            )

        # Use the categories learned from training.
        one_hot_encoded = self.one_hot_encoder_.transform(
            one_hot_data
        )

        return one_hot_encoded, frequency_data


    def fit_transform(self, train_df):
        """Fit on training data and immediately transform it."""
        self.fit(train_df)
        return self.transform(train_df)


    def get_feature_names(self):
        """Return names for all encoded categorical features."""
        if not self.is_fitted_:
            raise ValueError(
                "Encoder must be fitted before getting feature names."
            )

        one_hot_names = list(
            self.one_hot_encoder_.get_feature_names_out(
                self.one_hot_cols
            )
        )

        frequency_names = [
            f"{col}_freq"
            for col in self.frequency_cols
        ]

        return one_hot_names + frequency_names

### Encoder Test

A temporary subset of the raw transaction and identity data is used to verify that the encoder can fit and transform data successfully.

In [67]:
# Create a temporary raw merged dataset for testing the encoder.

identity_raw_cols = (
    ["TransactionID"]
    + [f"id_{i}" for i in range(12, 39)]
    + ["DeviceType", "DeviceInfo"]
)

raw_transaction_cats = pd.concat(
    [
        transaction_ids.reset_index(drop=True),
        transaction_df[transaction_cat_cols].reset_index(drop=True)
    ],
    axis=1
)

encoder_test_df = raw_transaction_cats.merge(
    identity_df[identity_raw_cols],
    on="TransactionID",
    how="left"
)

encoder_test_df.shape

(590540, 50)

In [68]:
# Use a temporary subset to test the encoder.

smoke_train = encoder_test_df.iloc[:100000].copy()
smoke_test = encoder_test_df.iloc[100000:101000].copy()

test_encoder = CategoricalEncoder()

test_encoder.fit(smoke_train)

smoke_one_hot, smoke_freq = test_encoder.transform(smoke_test)

In [69]:
# Check the encoder test results.

print("Fitted:", test_encoder.is_fitted_)
print("One-hot shape:", smoke_one_hot.shape)
print("Frequency shape:", smoke_freq.shape)
print("Frequency columns:", list(smoke_freq.columns))

Fitted: True
One-hot shape: (1000, 615)
Frequency shape: (1000, 7)
Frequency columns: ['card1_freq', 'id_19_freq', 'id_20_freq', 'id_21_freq', 'id_25_freq', 'id_26_freq', 'id_33_freq']


In [70]:
# Check how many encoded categorical features were created.

feature_names = test_encoder.get_feature_names()

print("Total encoded features:", len(feature_names))
print(feature_names[:20])

Total encoded features: 622
['ProductCD_C', 'ProductCD_H', 'ProductCD_R', 'ProductCD_S', 'ProductCD_W', 'card4_MISSING', 'card4_american express', 'card4_discover', 'card4_mastercard', 'card4_visa', 'card6_grouped_MISSING', 'card6_grouped_OTHER', 'card6_grouped_credit', 'card6_grouped_debit', 'addr2_101.0', 'addr2_102.0', 'addr2_13.0', 'addr2_14.0', 'addr2_15.0', 'addr2_16.0']


### Test Result

The reusable categorical encoder successfully fitted on a temporary training subset and transformed unseen rows without refitting.

The test produced 615 one-hot features and 7 frequency-encoded features, confirming that the encoder learns its category mappings from the data provided to `fit()` and applies those mappings consistently during `transform()`.

